# Pulse Edge — Gemma 3 1B LoRA fine-tune

Runs on a free Colab T4. ~30 min end-to-end. Produces a Q4_K_M GGUF the app can side-load.

In [ ]:
!pip install -q unsloth datasets transformers accelerate peft trl bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/gemma-3-1b-it-bnb-4bit',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)

In [ ]:
# Mix the public datasets + our persona examples.
import json
from datasets import load_dataset, Dataset, concatenate_datasets

persona = []
with open('/content/persona_examples.jsonl') as f:
    for line in f:
        persona.append(json.loads(line))
persona_ds = Dataset.from_list(persona)

med = load_dataset('medalpaca/medical_meadow_medqa', split='train[:1000]')

def to_chat(example):
    return {'messages': [
        {'role': 'user', 'content': example['input']},
        {'role': 'assistant', 'content': example['output']},
    ]}
med = med.map(to_chat, remove_columns=med.column_names)

full = concatenate_datasets([persona_ds, med]).shuffle(seed=42)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = full,
    dataset_text_field = 'messages',
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)
trainer.train()

In [ ]:
# Save merged + Q4_K_M GGUF for fllama / flutter_gemma.
model.save_pretrained_merged('outputs/merged', tokenizer, save_method='merged_16bit')
model.save_pretrained_gguf('outputs/gguf', tokenizer, quantization_method='q4_k_m')